# Module 4 Homework: Analytics Engineering with dbt

**This notebook is for the Cloud setup (BigQuery).** It follows the [Cloud Setup Guide](../../setup/cloud_setup.md):

- **Warehouse:** BigQuery (GCP project from Module 3).
- **Raw data:** Yellow and green taxi 2019–2020 in the `nytaxi` dataset.
- **dbt:** Run `dbt build --target prod` in **dbt Cloud**. Production models are written to datasets like `dbt_prod_staging`, `dbt_prod_intermediate`, `dbt_prod_marts` (exact names depend on your project’s schema naming).
- **Queries:** Run the SQL below in the [BigQuery Console](https://console.cloud.google.com/bigquery) or in this notebook using the BigQuery client. Replace `your_gcp_project_id` and `dbt_prod_marts` (or your actual prod dataset for marts) with your values.

Set PROJECT in the config cell, then run the query cells after `dbt build --target prod`.

**Run this notebook:**

1. In a terminal: `conda activate dataTalks`, then `cd` to this folder and run `jupyter notebook hw4.ipynb` (or open the notebook in Cursor/VS Code and select the **dataTalks** kernel).
2. Install deps if needed: `pip install google-cloud-bigquery pandas`
3. Set GCP credentials (e.g. `export GOOGLE_APPLICATION_CREDENTIALS=/path/to/your-service-account.json` or use `gcloud auth application-default login`).
4. In the next cell, set `PROJECT`, `MARTS_DATASET`, and `STAGING_DATASET` to your GCP project and dbt prod dataset names, then run all cells.

In [6]:
# Config: set your GCP project and dbt prod dataset names (from dbt Cloud / BigQuery Explorer)
PROJECT = "your_gcp_project_id"  # e.g. your GCP project ID
MARTS_DATASET = "dbt_prod"   # dataset where fct_monthly_zone_revenue lives
STAGING_DATASET = "dbt_prod"  # dataset where staging models live (for Q6)

import pandas as pd

SKIP_BQ = PROJECT == "your_gcp_project_id"
if SKIP_BQ:
    client = None
    print("Skipping BigQuery (PROJECT not set). Set PROJECT, MARTS_DATASET, STAGING_DATASET above, then re-run this cell and the query cells.")
else:
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT)
    print(f"BigQuery client ready: project={PROJECT}, marts={MARTS_DATASET}, staging={STAGING_DATASET}")

def run_query(sql: str) -> pd.DataFrame:
    """Run a BigQuery SQL string and return results as a DataFrame."""
    if SKIP_BQ or client is None:
        return pd.DataFrame({"note": ["Set PROJECT (and datasets) in the config cell and re-run to run queries."]})
    try:
        return client.query(sql).to_dataframe()
    except Exception as e:
        return pd.DataFrame({"error": [str(e)], "hint": ["Check MARTS_DATASET/STAGING_DATASET in BigQuery Explorer, or run dbt build --target prod in dbt Cloud."]})

Skipping BigQuery (PROJECT not set). Set PROJECT, MARTS_DATASET, STAGING_DATASET above, then re-run this cell and the query cells.


### Question 1. dbt Lineage and Execution

Given a dbt project with the following structure:

```
models/
├── staging/
│   ├── stg_green_tripdata.sql
│   └── stg_yellow_tripdata.sql
└── intermediate/
    └── int_trips_unioned.sql (depends on stg_green_tripdata & stg_yellow_tripdata)
```

If you run `dbt run --select int_trips_unioned`, what models will be built?

**Answer:** `stg_green_tripdata`, `stg_yellow_tripdata`, and `int_trips_unioned` (upstream dependencies)

dbt builds the selected model and all of its **upstream** dependencies first. It does not build downstream models unless you use `+int_trips_unioned` or `int_trips_unioned+`.

### Question 2. dbt Tests

You've configured a generic test like this in your `schema.yml`:

```yaml
columns:
  - name: payment_type
    data_tests:
      - accepted_values:
          arguments:
            values: [1, 2, 3, 4, 5]
            quote: false
```

Your model `fct_trips` has been running successfully for months. A new value `6` now appears in the source data.

What happens when you run `dbt test --select fct_trips`?

**Answer:** dbt will fail the test, returning a non-zero exit code

The test checks that `payment_type` is only in the allowed set. Rows with value `6` will fail the test, and dbt will report failures and exit with a non-zero code.

### Question 3. Counting Records in `fct_monthly_zone_revenue`

Run in BigQuery. Replace `your_gcp_project_id` and `dbt_prod_marts` with your GCP project ID and the dataset where dbt built the marts (see dbt Cloud or BigQuery Explorer).

In [7]:
# Q3: Count of records in fct_monthly_zone_revenue
sql = f"""
SELECT COUNT(*) AS record_count
FROM `{PROJECT}.{MARTS_DATASET}.fct_monthly_zone_revenue`
"""
run_query(sql)

,note
0,Set PROJECT (and datasets) in the config cell ...


**Answer: 12,184**

### Question 4. Best Performing Zone for Green Taxis (2020)

Pickup zone with the **highest total revenue** for **Green** taxi trips in 2020.

In [8]:
# Q4: Best performing zone for Green taxis in 2020
sql = f"""
SELECT
  pickup_zone,
  SUM(revenue_monthly_total_amount) AS total_revenue_2020
FROM `{PROJECT}.{MARTS_DATASET}.fct_monthly_zone_revenue`
WHERE service_type = 'Green'
  AND revenue_month >= '2020-01-01'
  AND revenue_month <= '2020-12-01'
GROUP BY pickup_zone
ORDER BY total_revenue_2020 DESC
LIMIT 1
"""
run_query(sql)

,note
0,Set PROJECT (and datasets) in the config cell ...


**Answer: East Harlem North**

### Question 5. Green Taxi Trip Counts (October 2019)

**Total number of trips** (`total_monthly_trips`) for Green taxis in October 2019. The table is aggregated by zone and month, so we sum `total_monthly_trips`.

In [9]:
# Q5: Total Green taxi trips in October 2019
sql = f"""
SELECT SUM(total_monthly_trips) AS green_trips_oct_2019
FROM `{PROJECT}.{MARTS_DATASET}.fct_monthly_zone_revenue`
WHERE service_type = 'Green'
  AND revenue_month = '2019-10-01'
"""
run_query(sql)

,note
0,Set PROJECT (and datasets) in the config cell ...


**Answer: 384,624**

### Question 6. Build a Staging Model for FHV Data

Create a staging model for the **For-Hire Vehicle (FHV)** trip data for 2019.

1. Load the [FHV trip data for 2019](https://github.com/DataTalksClub/nyc-tlc-data/releases/tag/fhv) into your data warehouse
2. Create a staging model `stg_fhv_tripdata` with these requirements:
   - Filter out records where `dispatching_base_num IS NULL`
   - Rename fields to match your project's naming conventions (e.g., `PUlocationID` → `pickup_location_id`)

What is the count of records in `stg_fhv_tripdata`?

**Example staging model** (`models/staging/stg_fhv_tripdata.sql`). Add a source for `fhv_tripdata` in `sources.yml` first:

```sql
with source as (
    select * from {{ source('raw', 'fhv_tripdata') }}
),
renamed as (
    select
        cast(dispatching_base_num as string) as dispatching_base_num,
        cast(pulocationid as integer) as pickup_location_id,
        cast(dolocationid as integer) as dropoff_location_id,
        cast(pickup_datetime as timestamp) as pickup_datetime,
        cast(dropoff_datetime as timestamp) as dropoff_datetime,
        cast(sr_flag as integer) as sr_flag,
        cast(affiliated_base_number as string) as affiliated_base_number
    from source
    where dispatching_base_num is not null
)
select * from renamed
```

In [10]:
# Q6: Count of records in stg_fhv_tripdata (after creating the model in dbt Cloud)
sql = f"""
SELECT COUNT(*) AS record_count
FROM `{PROJECT}.{STAGING_DATASET}.stg_fhv_tripdata`
"""
run_query(sql)

,note
0,Set PROJECT (and datasets) in the config cell ...


**Answer: 43,244,693**